In [25]:
import pandas as pd
import glob
import os

# Define the folder where your data is stored (update this if your folder name is different)
data_folder = "C:/Users/haris/Downloads/Harish/kaggle Hackathon/train/"# or '.' if the files are in your current directory

# Find all horizontal well files in the folder
horizontal_files = glob.glob(f"{data_folder}*__horizontal_well.csv")

# Create an empty list to store the summaries for all wells
all_wells_summary = []

print(f"Found {len(horizontal_files)} wells. Processing...")

# Loop through each found file
for hw_path in horizontal_files:
    try:
        # Extract the unique well ID from the filename (e.g., 'fde20ecf' from 'fde20ecf__horizontal_well.csv')
        base_name = os.path.basename(hw_path)
        well_id = base_name.split('__')[0]
        
        # Construct the paths for the matching typewell and image files
        tw_path = os.path.join(data_folder, f"{well_id}__typewell.csv")
        png_path = os.path.join(data_folder, f"{well_id}.png")
        
        # Load the CSV data
        hw_df = pd.read_csv(hw_path)
        
        # Check if typewell file exists before reading (some test wells might not have them)
        if os.path.exists(tw_path):
            tw_df = pd.read_csv(tw_path)
            tw_shape = tw_df.shape
            tw_formations = tw_df['Geology'].dropna().unique().tolist()
        else:
            tw_shape = "Missing"
            tw_formations = []
            
        # Check if the PNG exists
        has_png = os.path.exists(png_path)

        # Compile the summary dictionary for this specific well
        well_info = {
            'Well_ID': well_id,
            'HW_Rows': hw_df.shape[0],
            'Missing_TVT': hw_df['TVT'].isnull().sum() if 'TVT' in hw_df.columns else "N/A",
            'Missing_TVT_input': hw_df['TVT_input'].isnull().sum() if 'TVT_input' in hw_df.columns else "N/A",
            'HW_GR_Min': hw_df['GR'].min(),
            'HW_GR_Max': hw_df['GR'].max(),
            'HW_MD_Min': hw_df['MD'].min(),
            'HW_MD_Max': hw_df['MD'].max(),
            'TW_Shape': tw_shape,
            'TW_Formations_Count': len(tw_formations),
            'Has_PNG': has_png
        }
        
        # Add this well's info to our master list
        all_wells_summary.append(well_info)
        
    except Exception as e:
        print(f"Error processing well {well_id}: {e}")

# Convert the master list of dictionaries into a Pandas DataFrame for easy viewing
summary_df = pd.DataFrame(all_wells_summary)

# Display the first few rows of the final summary table
print("\n--- Processing Complete ---")
display(summary_df.head()) # Use print(summary_df.head()) if not in a Jupyter/Kaggle notebook

Found 773 wells. Processing...

--- Processing Complete ---


,Well_ID,HW_Rows,Missing_TVT,Missing_TVT_input,HW_GR_Min,HW_GR_Max,HW_MD_Min,HW_MD_Max,TW_Shape,TW_Formations_Count,Has_PNG
0,000d7d20,5278,0,3836,31.765827,217.352257,11467.0,16744.0,"(1296, 3)",10,True
1,00bbac68,7559,0,6014,37.132148,219.806788,11578.0,19136.0,"(1946, 3)",6,True
2,00e12e8b,6384,0,4301,21.892866,184.360973,10456.0,16839.0,"(2556, 3)",10,True
3,015fe0d2,5950,0,4296,27.806823,201.041336,11834.0,17783.0,"(1265, 3)",6,True
4,01869cd4,6850,0,5557,31.337161,182.482017,11256.0,18105.0,"(1052, 3)",10,True


In [26]:
import os
print("Python is currently looking in this folder:")
print(os.getcwd())

Python is currently looking in this folder:
c:\Users\haris\Downloads\Harish\kaggle Hackathon\train


In [20]:
import pandas as pd
import glob
import os

# 1. Point to your folder (Update this to your exact path)
data_folder = "C:/Users/haris/Downloads/Harish/kaggle Hackathon/train/"

# Find all the horizontal well files
horizontal_files = glob.glob(f"{data_folder}*__horizontal_well.csv")

# We will store the dataframes in this list temporarily
all_wells_data = []

print(f"Starting to merge {len(horizontal_files)} files... This might take a minute.")

for hw_path in horizontal_files:
    try:
        # Extract the unique well ID
        base_name = os.path.basename(hw_path)
        well_id = base_name.split('__')[0]
        
        # Load the individual CSV file
        df = pd.read_csv(hw_path)
        
        # CRITICAL STEP: Add a column so we know which well this row belongs to!
        df['WELL_ID'] = well_id 
        
        # Add this dataframe to our list
        all_wells_data.append(df)
        
    except Exception as e:
        print(f"Error loading well {well_id}: {e}")

# 2. Stack them all together vertically into one giant dataframe
print("Stitching the files together...")
master_df = pd.concat(all_wells_data, ignore_index=True)

# 3. Save the giant dataframe to a single CSV file
output_filename = 'MASTER_training_data.csv'
print(f"Saving to {output_filename}...")
master_df.to_csv(output_filename, index=False)

print("\nSuccess! You now have a master training file.")
print(f"Total Rows: {master_df.shape[0]}")
print(f"Total Columns: {master_df.shape[1]}")

Starting to merge 773 files... This might take a minute.
Stitching the files together...
Saving to MASTER_training_data.csv...

Success! You now have a master training file.
Total Rows: 5092255
Total Columns: 14


In [21]:
import pandas as pd

# Load the file directly via Python
file_path = "MASTER_training_data.csv"
print("Counting rows...")
df = pd.read_csv(file_path)

# Print the true shape of the data
print(f"The actual number of rows in your CSV is: {df.shape[0]}")

Counting rows...
The actual number of rows in your CSV is: 5092255


In [22]:
import pandas as pd

# Load your master file
file_path = "MASTER_training_data.csv"
print("Loading master dataset for Deep Anomaly Scan...")
df = pd.read_csv(file_path)

print("\n--- DEEP SCAN REPORT ---")

# 1. Check for Backwards Drilling (MD should always increase within a well)
# We sort by well and depth just in case the rows got shuffled
df = df.sort_values(by=['WELL_ID', 'MD'])
df['MD_diff'] = df.groupby('WELL_ID')['MD'].diff()

backwards_drilling = (df['MD_diff'] < 0).sum()
if backwards_drilling > 0:
    print(f"⚠️ WARNING: Found {backwards_drilling} rows where Measured Depth (MD) went backwards!")
else:
    print("✅ No backwards drilling anomalies found. MD always increases.")

# 2. Check for Extreme Gamma Ray (GR) Spikes
gr_max = df['GR'].max()
if gr_max > 500: # 500 is a safe threshold; normal is usually < 250
    print(f"⚠️ WARNING: Found highly abnormal Gamma Ray readings! The maximum GR is {gr_max:.2f}")
else:
    print(f"✅ Gamma Ray readings look physically normal (Max: {gr_max:.2f}).")

# 3. Check for exact duplicate coordinates
# The drill shouldn't record the exact same X, Y, Z, and MD multiple times
duplicates = df.duplicated(subset=['WELL_ID', 'X', 'Y', 'Z', 'MD']).sum()
if duplicates > 0:
    print(f"⚠️ WARNING: Found {duplicates} completely duplicated rows!")
else:
    print("✅ No duplicated coordinates found.")

print("\nScan Complete!")

Loading master dataset for Deep Anomaly Scan...

--- DEEP SCAN REPORT ---
✅ No backwards drilling anomalies found. MD always increases.
✅ Gamma Ray readings look physically normal (Max: 487.03).
✅ No duplicated coordinates found.

Scan Complete!


In [23]:
pip install fastdtw scipy


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas as pd
import numpy as np
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import glob
import os

# 1. Define the folder
data_folder = "C:\\Users\\haris\\Downloads\\Harish\\kaggle Hackathon\\train\\MASTER_training_data.csv"

# 2. Let Python safely find the files instead of hardcoding the ID
search_path = os.path.join(data_folder, "*__horizontal_well.csv")
available_files = glob.glob(search_path)

if not available_files:
    print(f"❌ Error: Python could not find any files in: {data_folder}")
    print("Double check that the folder path is exactly correct.")
else:
    # Grab the very first file Python finds and extract its ID automatically
    hw_path = available_files[0]
    base_name = os.path.basename(hw_path)
    well_id = base_name.split('__')[0]
    
    # Automatically build the safe path for the Typewell
    tw_path = os.path.join(data_folder, f"{well_id}__typewell.csv")
    
    print(f"✅ Successfully found files for Well {well_id}!")
    
    # 3. Load the data
    hw_df = pd.read_csv(hw_path)
    tw_df = pd.read_csv(tw_path)
    
    # Extract the Gamma Ray signals as arrays (dropping blanks)
    hw_gr_signal = hw_df['GR'].dropna().values
    tw_gr_signal = tw_df['GR'].dropna().values

    print(f"Horizontal Signal Length: {len(hw_gr_signal)}")
    print(f"Typewell Signal Length: {len(tw_gr_signal)}")

    # 4. Perform Dynamic Time Warping
    print("\nCalculating DTW Alignment... (This may take a few seconds)")
    distance, path = fastdtw(hw_gr_signal, tw_gr_signal, dist=euclidean)

    print(f"\n✅ DTW Complete!")
    print(f"Total Alignment Distance (Error): {distance:.2f}")

    # 5. Show how the algorithm matched the points
    print("\nFirst 5 alignment pairs (Horizontal Index -> Typewell Index):")
    for i in range(5):
        horizontal_idx = path[i][0]
        typewell_idx = path[i][1]
        print(f"Horizontal Row {horizontal_idx} matches Typewell Row {typewell_idx}")

❌ Error: Python could not find any files in: C:\Users\haris\Downloads\Harish\kaggle Hackathon\train\MASTER_training_data.csv
Double check that the folder path is exactly correct.
